# Finance Complaint Triage: build the real-complaint dataset

This notebook gets the archived CFPB complaint data, cleans and labels it, and splits it into
train, validation and test (proposal v2, data plan steps 1 to 5 and 9).

| Step | What happens |
| --- | --- |
| 1. Get | Download the pinned Hugging Face snapshot `Mouwiya/cfpb-consumer-complaints` (about 1 GB, 16.6M rows) |
| 2. Look | Narratives per quarter, and the product rename on 2023-08-25 that sets our date window |
| 3. Filter | 2023-08-25 to 2026-06-17, narrative of 50 to 400 words, `b'...'` artifacts unwrapped |
| 4. Labels | Top 10 products, each with its 5 most common issues plus `other` |
| 5. Clean | Exact duplicates, non-English text and near duplicates (MinHash, Jaccard 0.9 or above) removed |
| 6. Sample | 135 complaints per product, 1,350 in total |
| 7. Privacy | Emails, SSNs, phone numbers and card or account numbers masked as `XXXX` |
| 8. Split | Seed 42, stratified by product: 111 / 9 / 15 per product = **1,110 / 90 / 150** |
| 9. Verify | Checks that you got exactly the team's frozen split |

**Runtime:** about 15 minutes on a free Colab CPU runtime, most of it the download and step 3.
No GPU is needed.

**Outputs** go to `OUT_DIR`: `real_train.jsonl`, `real_val.jsonl`, `real_test.jsonl`,
`labels.json` and `build_log.json`.

> `real_test.jsonl` is the closed test set. Do not open it to tune prompts.

## 0. Setup

In [ ]:
!pip -q install datasketch huggingface_hub pyarrow

In [ ]:
# ---- configuration: change OUT_DIR if you want the files somewhere else ----
SAVE_TO_DRIVE = False          # True: mount Google Drive and write the outputs there
OUT_DIR = "data"               # relative to /content, or to MyDrive when SAVE_TO_DRIVE
DATA_DIR = "raw/cfpb_mouwiya"  # where the snapshot is downloaded (reused if already there)

# ---- fixed by the proposal: change these and you no longer get the team's split ----
HF_REPO = "Mouwiya/cfpb-consumer-complaints"
REVISION = "038f8f8b18879c9384abfee5d0b685b00c03b3cf"   # snapshot of 2026-07-06
SINCE, UNTIL = "2023-08-25", "2026-06-17"                 # stable product names, last narrative
MIN_WORDS, MAX_WORDS = 50, 400
N_PRODUCTS, MIN_PER_PRODUCT, N_ISSUES = 10, 170, 5
PER_PRODUCT = 135
N_TRAIN, N_VAL, N_TEST = 111, 9, 15                       # per product
CANDIDATES = 1500                                          # random candidates kept per product
SEED = 42

assert N_TRAIN + N_VAL + N_TEST == PER_PRODUCT

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = os.path.join("/content/drive/MyDrive", OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
print("outputs ->", os.path.abspath(OUT_DIR))

## 1. Get the data

The CFPB stopped publishing complaint narratives on 14 August 2026, so we use an archived
copy. The download is pinned to one revision, so everyone gets the same bytes. Using this
archived snapshot still needs instructor approval.

In [ ]:
import glob
import pyarrow.parquet as pq
from huggingface_hub import snapshot_download

snapshot_download(repo_id=HF_REPO, repo_type="dataset", revision=REVISION,
                  allow_patterns=["data/*.parquet", "metadata/*", "README.md"],
                  local_dir=DATA_DIR, max_workers=8)
FILES = sorted(glob.glob(os.path.join(DATA_DIR, "data", "*.parquet")))
n_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in FILES)
print(f"{len(FILES)} parquet files, {n_rows:,} rows")
assert len(FILES) == 67 and n_rows == 16_558_052, "not the expected snapshot"

## 2. A quick look: why the window starts on 2023-08-25

On 2023-08-25 the CFPB switched to new product and issue names. For example, "Credit card or
prepaid card" was split into "Credit card" and "Prepaid card". Starting the window on that
date means every complaint uses the same names, so no mapping table is needed. Narratives
then thin out through 2026 and stop on 2026-06-17.

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds


def date_filter(d, since=None, until=None):
    # The snapshot mixes timestamp[us] and timestamp[us, tz=UTC] across files,
    # so each file gets a filter built for its own date type.
    typ = d.schema.field("date_received").type
    lit = lambda ts: pa.scalar(ts.tz_localize("UTC") if typ.tz else ts, typ)
    f = None
    if since is not None:
        f = ds.field("date_received") >= lit(since)
    if until is not None:
        g = ds.field("date_received") < lit(until)
        f = g if f is None else f & g
    return f


frames = []
for f in FILES:
    d = ds.dataset(f, format="parquet")
    t = d.to_table(columns=["date_received", "product", "consumer_complaint_narrative"],
                   filter=date_filter(d, since=pd.Timestamp("2022-06-01")))
    text = t.column("consumer_complaint_narrative")
    has = (pc.is_valid(text) if (pa.types.is_string(text.type) or pa.types.is_large_string(text.type))
           else pa.array([False] * len(t)))
    x = pd.DataFrame({"date": t.column("date_received").to_pandas(),
                      "product": t.column("product").to_pandas(),
                      "has_text": has.to_numpy(zero_copy_only=False)})
    if x["date"].dt.tz is not None:
        x["date"] = x["date"].dt.tz_convert(None)
    frames.append(x)
recent = pd.concat(frames, ignore_index=True)
del frames

q = recent.groupby(recent["date"].dt.to_period("Q"))["has_text"].agg(complaints="size", with_text="sum")
q["share_with_text"] = (q["with_text"] / q["complaints"]).round(3)
display(q)

names = recent[recent["has_text"]].groupby("product")["date"].agg(first="min", last="max", n="size")
display(names.sort_values("first"))

## 3. Helpers

In [ ]:
import ast, codecs, hashlib, json, re, warnings
from collections import Counter
import numpy as np

COLS = ["date_received", "product", "issue", "sub_issue",
        "consumer_complaint_narrative", "complaint_id"]

# Very common English words. A narrative with fewer than MIN_EN_SHARE of these among its
# words is treated as non-English. Words shared with Spanish (no, me, a) are left out.
EN_WORDS = set('''the and to of i in my was that it for on is with have not this they
be as had but at by from or an are their we were been would so you will all
did has them do there which about when what account'''.split())
MIN_EN_SHARE = 0.15

PII_PATTERNS = {
    "email": re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "ssn": re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "phone": re.compile(r"(?:\+?1[ .-]?)?(?:\(\d{3}\)\s?|\b\d{3}[ .-])\d{3}[ .-]\d{4}\b"),
    "card_or_account": re.compile(r"\b\d(?:[ -]?\d){7,18}\b"),
}
BYTES_REPR = re.compile("^b(['\"])")


def word_count(arr):
    return pc.list_value_length(pc.utf8_split_whitespace(arr))


def unwrap(text):
    # About 3,000 narratives are stored as the text of a Python bytes object:
    # b'I called ...\\n...' with escaped newlines and UTF-8 bytes, some without the
    # closing quote. Turn them back into plain text. Returns (text, changed).
    t = text.strip()
    m = BYTES_REPR.match(t)
    if not m:
        return text, False
    # a backslash before XXXX is an "invalid escape": Python keeps it and warns, so mute that
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", (SyntaxWarning, DeprecationWarning))
        try:
            return ast.literal_eval(t).decode("utf-8", errors="replace"), True
        except (ValueError, SyntaxError):
            body = t[2:-1] if t.endswith(m.group(1)) else t[2:]
            body = re.sub(r"\\+$", lambda s: s.group()[: len(s.group()) // 2 * 2], body)
            raw = codecs.escape_decode(body.encode("utf-8", "backslashreplace"))[0]
            return raw.decode("utf-8", errors="replace"), True


def en_share(text):
    toks = [t for t in re.findall(r"[a-z]+", text.lower()) if t != "xxxx"]
    return sum(t in EN_WORDS for t in toks) / len(toks) if toks else 0.0


def shingles(text, n=3):
    w = re.findall(r"[a-z0-9]+", text.lower())
    return {" ".join(w[i:i + n]) for i in range(max(1, len(w) - n + 1))}


def scrub(text, counts):
    for name, pat in PII_PATTERNS.items():
        text, k = pat.subn("XXXX", text)
        counts[name] += k
    return text


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


print("unwrap test:", unwrap("b'It" + "\\" + "xe2" + "\\" + "x80" + "\\" + "x99s late." + "\\" + "nCall me.'"))

## 4. Filter and keep random candidates

This step streams the window in batches, keeping complaints whose narrative has 50 to 400
words. It counts products and issues over all eligible complaints, then keeps 1,500
**seeded random candidates per product**: the lowest values of a hash of the complaint ID.
That way the whole 1.65M pool never has to sit in memory, and the same candidates come out
every time.

In [ ]:
since = pd.Timestamp(SINCE)
until = pd.Timestamp(UNTIL) + pd.Timedelta(days=1)
hash_key = f"cfpbseed{SEED:08d}"[:16]
log, prod_n, issue_n, cand = Counter(), Counter(), Counter(), {}

for fi, f in enumerate(FILES, 1):
    d = ds.dataset(f, format="parquet")
    for batch in d.to_batches(columns=COLS, filter=date_filter(d, since, until), batch_size=200_000):
        log["rows_in_window"] += batch.num_rows
        text = batch.column("consumer_complaint_narrative")
        if not (pa.types.is_string(text.type) or pa.types.is_large_string(text.type)):
            assert text.null_count == len(text)      # a file with no narratives at all
            continue
        has = pc.fill_null(pc.greater(pc.utf8_length(pc.utf8_trim_whitespace(text)), 0), False)
        log["with_text"] += pc.sum(has).as_py() or 0
        b = batch.filter(has)
        if b.num_rows == 0:
            continue
        w = word_count(b.column("consumer_complaint_narrative"))
        b = b.filter(pc.and_(pc.greater_equal(w, MIN_WORDS), pc.less_equal(w, MAX_WORDS)))
        df = b.to_pandas()
        if df["date_received"].dt.tz is not None:
            df["date_received"] = df["date_received"].dt.tz_convert(None)
        un = df["consumer_complaint_narrative"].map(unwrap)
        changed = un.map(lambda r: r[1]).to_numpy(dtype=bool)
        if changed.any():
            log["unwrapped_bytes_repr"] += int(changed.sum())
            df["consumer_complaint_narrative"] = un.map(lambda r: r[0])
        df["words"] = df["consumer_complaint_narrative"].str.split().str.len()
        df = df[df["words"].between(MIN_WORDS, MAX_WORDS)].copy()   # unwrapping can change counts
        log["eligible_50_400"] += len(df)
        df["product_raw"] = df["product"].fillna("(missing)")
        df["product"] = df["product_raw"]
        df["issue"] = df["issue"].fillna("(missing)")
        prod_n.update(df["product"])
        issue_n.update(zip(df["product"], df["issue"]))
        df["_key"] = pd.util.hash_pandas_object(df["complaint_id"], index=False,
                                                hash_key=hash_key).to_numpy()
        for prod, g in df.groupby("product"):
            g = pd.concat([cand[prod], g]) if prod in cand else g
            cand[prod] = g.nsmallest(CANDIDATES, "_key")
    if fi % 10 == 0 or fi == len(FILES):
        print(f"file {fi}/{len(FILES)}: {log['rows_in_window']:,} in window, "
              f"{log['eligible_50_400']:,} eligible", flush=True)

print(dict(log))

## 5. Fix the labels

This keeps the 10 products with the most eligible narratives, provided each has at least 170.
For each product it keeps the 5 most common issues, and every other issue becomes `other`.

In [ ]:
ranked = [p for p, n in prod_n.most_common() if p != "(missing)"]
chosen = [p for p in ranked if prod_n[p] >= MIN_PER_PRODUCT][:N_PRODUCTS]
assert len(chosen) == N_PRODUCTS, f"only {len(chosen)} products have >= {MIN_PER_PRODUCT}"

labels = {"products": []}
for p in chosen:
    iss = sorted(((n, i) for (pp, i), n in issue_n.items() if pp == p), reverse=True)
    tot = sum(n for n, _ in iss)
    labels["products"].append({
        "product": p,
        "issues": [i for _, i in iss[:N_ISSUES]] + ["other"],
        "eligible_in_window": prod_n[p],
        "top_issue_share": round(sum(n for n, _ in iss[:N_ISSUES]) / tot, 4),
    })
labels["out_of_scope"] = "out_of_scope"
labels["window"] = {"since": SINCE, "until": UNTIL}
top_issues = {d["product"]: set(d["issues"][:-1]) for d in labels["products"]}

display(pd.DataFrame([{k: d[k] for k in ["product", "eligible_in_window", "top_issue_share"]}
                      for d in labels["products"]]))

## 6. Clean the candidates

This removes exact duplicates (after trimming and lowercasing), then non-English narratives,
then near duplicates (MinHash on word 3-grams, Jaccard 0.9 or above). Candidates are processed
in seeded order, so the first copy of a duplicate group is the one kept.

In [ ]:
from datasketch import MinHash, MinHashLSH

pool = pd.concat([cand[p] for p in chosen]).sort_values("_key")
log["candidates"] = len(pool)
pool["_norm"] = pool["consumer_complaint_narrative"].str.strip().str.lower()
n0 = len(pool)
pool = pool.drop_duplicates("_norm")
log["dropped_exact_duplicate"] = n0 - len(pool)

pool["_en"] = pool["consumer_complaint_narrative"].map(en_share)
log["dropped_non_english"] = int((pool["_en"] < MIN_EN_SHARE).sum())
pool = pool[pool["_en"] >= MIN_EN_SHARE]

lsh, keep = MinHashLSH(threshold=0.9, num_perm=128), []
for cid, t in zip(pool["complaint_id"], pool["consumer_complaint_narrative"]):
    m = MinHash(num_perm=128, seed=SEED)
    for s in shingles(t):
        m.update(s.encode())
    if lsh.query(m):
        keep.append(False)
    else:
        lsh.insert(str(cid), m)
        keep.append(True)
log["dropped_near_duplicate"] = len(keep) - sum(keep)
pool = pool[np.array(keep)]
print({k: log[k] for k in ["candidates", "dropped_exact_duplicate",
                           "dropped_non_english", "dropped_near_duplicate"]})

## 7. Sample, privacy pass and split

This takes the first 135 clean candidates of each product and masks any personal data left in
them. It then splits each product with seed 42 into 111 train, 9 validation and 15 test.

In [ ]:
rng = np.random.default_rng(SEED)
parts, pii = [], Counter()
for p in chosen:
    g = pool[pool["product"] == p].head(PER_PRODUCT).copy()
    assert len(g) == PER_PRODUCT, f"{p}: only {len(g)} usable candidates"
    g["consumer_complaint_narrative"] = [scrub(t, pii) for t in g["consumer_complaint_narrative"]]
    order = rng.permutation(len(g))
    split = np.empty(len(g), dtype=object)
    split[order[:N_TEST]] = "test"
    split[order[N_TEST:N_TEST + N_VAL]] = "val"
    split[order[N_TEST + N_VAL:]] = "train"
    g["split"] = split
    parts.append(g)
sample = pd.concat(parts)
sample["issue_raw"] = sample["issue"]
sample["issue"] = [i if i in top_issues[p] else "other"
                   for p, i in zip(sample["product"], sample["issue"])]

print("privacy pass, items masked:", dict(pii) or "none")
display(sample.groupby(["product", "split"]).size().unstack()[["train", "val", "test"]])

## 8. Format and save

Each line of the output files is one complaint in this format:

| Field | Meaning |
| --- | --- |
| `complaint_id` | CFPB complaint ID |
| `date_received` | `YYYY-MM-DD` |
| `product` | One of the 10 labels in `labels.json`: the model's target |
| `issue` | One of that product's 5 issues, or `other`: the model's target |
| `product_raw`, `issue_raw`, `sub_issue` | The labels as the consumer chose them |
| `words` | Word count of the narrative |
| `narrative` | The cleaned complaint text: the model's input |

Rows are sorted by product, then complaint ID, so the files are byte-identical on every run.

In [ ]:
OUT_COLS = ["complaint_id", "date_received", "product", "issue", "product_raw",
            "issue_raw", "sub_issue", "words", "narrative"]
out = sample.rename(columns={"consumer_complaint_narrative": "narrative"})
out["date_received"] = out["date_received"].dt.strftime("%Y-%m-%d")
out["words"] = out["words"].astype(int)

files = {}
for s in ["train", "val", "test"]:
    part = out[out["split"] == s].sort_values(["product", "complaint_id"])
    path = os.path.join(OUT_DIR, f"real_{s}.jsonl")
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for r in part[OUT_COLS].to_dict("records"):
            # missing values (sub_issue) become null; newer pandas gives NaN, which is not JSON
            r = {k: None if isinstance(v, float) and v != v else v for k, v in r.items()}
            f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    files[f"real_{s}.jsonl"] = {"rows": len(part), "sha256": sha256(path),
                                "ids": sorted(part["complaint_id"])}

with open(os.path.join(OUT_DIR, "labels.json"), "w", encoding="utf-8") as f:
    json.dump(labels, f, indent=2, ensure_ascii=False)
build_log = {
    "snapshot": {"repo": HF_REPO, "revision": REVISION},
    "config": {"since": SINCE, "until": UNTIL, "words": [MIN_WORDS, MAX_WORDS],
               "per_product": PER_PRODUCT, "split": [N_TRAIN, N_VAL, N_TEST], "seed": SEED},
    "counts": dict(log), "pii_masked": dict(pii), "products_chosen": chosen,
    "sample_other_share": round(float((out["issue"] == "other").mean()), 4),
    "files": {k: {"rows": v["rows"], "sha256": v["sha256"]} for k, v in files.items()},
}
with open(os.path.join(OUT_DIR, "build_log.json"), "w", encoding="utf-8") as f:
    json.dump(build_log, f, indent=2, ensure_ascii=False, default=str)
for k, v in files.items():
    print(f"{k:18} {v['rows']:5}  sha256 {v['sha256']}")

## 9. Verify: is this the team's frozen split?

The **ID check** is the one that matters: each split must contain exactly the same complaints as
the team's build. The **file hash** also matches when the library versions match the team's
build (pandas 3.0.6, pyarrow 26.0.0, datasketch 2.0.0). On other versions it can differ even
when the IDs match. If so, the IDs are what count, but share the build with the files from the
team's copy so everyone trains on identical bytes.

In [ ]:
EXPECTED = {
    "real_train.jsonl": {"rows": 1110,
        "ids_sha256": "b6b82fd9bb7810fcb11cf4d7acb7e14409c7957256fb800630d96b0c5313bfd1",
        "file_sha256": "076df0251f7c22f99c3246cf501acb629bd20a8211cdc2f69c8129d38db020c2"},
    "real_val.jsonl": {"rows": 90,
        "ids_sha256": "6059b064129d1a7914f4b0f99014b409be0065a6acb3be2ae58734ed937a21c6",
        "file_sha256": "7ea00d9c19ada7f40aa488313d9696901172d371cb05c3678bc847d2c7b17dd3"},
    "real_test.jsonl": {"rows": 150,
        "ids_sha256": "d92f9e68d6d6410fa99856846b1e1008d7daa5f9d30d9b9fdbecfe1020641fd9",
        "file_sha256": "38a5c275f3432c09d4aa8bf899b5a3f646a19837140b41eff90e618921bfbdaa"},
}
import pandas, pyarrow, datasketch
print("versions: pandas", pandas.__version__, "| pyarrow", pyarrow.__version__,
      "| datasketch", datasketch.__version__)
ok = True
for name, exp in EXPECTED.items():
    got = files[name]
    ids_hash = hashlib.sha256("\n".join(got["ids"]).encode()).hexdigest()
    ids_ok, file_ok = ids_hash == exp["ids_sha256"], got["sha256"] == exp["file_sha256"]
    ok &= ids_ok and got["rows"] == exp["rows"]
    print(f"{name:18} rows {got['rows']:5}  IDs {'match' if ids_ok else 'DIFFER'}  "
          f"file hash {'match' if file_ok else 'differs'}")
assert ok, "The split differs from the team's frozen split: check the snapshot and settings."
print("\nOK: this is the team's frozen split.")

## 10. Look at one formatted record (train only)

In [ ]:
with open(os.path.join(OUT_DIR, "real_train.jsonl"), encoding="utf-8") as f:
    rec = json.loads(next(f))
print(json.dumps({k: v for k, v in rec.items() if k != "narrative"}, indent=2))
print("\nnarrative:\n" + rec["narrative"][:600])

# The fields the model must produce for this complaint. Step 11 adds the teacher reply and the
# citations and writes the chat-format fine-tuning files.
target = {"product": rec["product"], "issue": rec["issue"], "reply": None, "citations": []}
print("\nmodel target:", json.dumps(target, ensure_ascii=False))

## 11. Build the fine-tuning files

This step gets the team's repository (teacher replies, out-of-scope examples, retrieved Ask CFPB
passages and `scripts/build_sft.py`) and builds the chat-format files for fine-tuning:

| File | Rows | What it holds |
| --- | --- | --- |
| `sft/cfpb_train.jsonl` | 1,200 | 1,110 real + 90 out of scope; half have guidance passages |
| `sft/cfpb_val.jsonl` | 100 | 90 real + 10 out of scope, all with passages |
| `sft/cfpb_test.jsonl` | 170 | 150 real + 20 out of scope, labels only (no reply): score with `eval/run.py` |

Each line has `messages` (system, user, assistant) in the format `trl`'s `SFTTrainer` reads
directly. The check below compares the files with the team's build.

In [ ]:
import shutil, subprocess

REPO_URL = "https://github.com/parulrewatkar0692/finance-complaint-triage"
EXPECTED_SFT = {
    "cfpb_train.jsonl": "f8d8e29db1b3e5a71da13422e75ea5797b28482d2cd43c8e43d7c2f1a073abcd",
    "cfpb_val.jsonl": "1abd516e93244ffde811b99641f56fea69d259a95d3dc466091037f32f911ec6",
    "cfpb_test.jsonl": "7f1b8762ce605298385365066af58b06f9a9d9481892f88c12eb8ae2fc73fa9e"
}

if os.path.exists(os.path.join("..", "scripts", "build_sft.py")):
    REPO_DIR = os.path.abspath("..")          # running inside a copy of the repository
else:
    REPO_DIR = os.path.abspath("finance-complaint-triage")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
for name in ["real_train.jsonl", "real_val.jsonl", "real_test.jsonl"]:
    src, dst = os.path.join(OUT_DIR, name), os.path.join(REPO_DIR, "data", name)
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
run = subprocess.run([sys.executable, os.path.join(REPO_DIR, "scripts", "build_sft.py")],
                     capture_output=True, text=True)
print(run.stdout, run.stderr)
run.check_returncode()

SFT_DIR = os.path.join(OUT_DIR, "sft")
os.makedirs(SFT_DIR, exist_ok=True)
manifest = json.load(open(os.path.join(REPO_DIR, "data", "sft", "manifest.json")))
for name in list(EXPECTED_SFT) + ["manifest.json"]:
    shutil.copy(os.path.join(REPO_DIR, "data", "sft", name), os.path.join(SFT_DIR, name))
shutil.copy(os.path.join(REPO_DIR, "data", "teacher", "review_sample.csv"), OUT_DIR)
same = all(manifest["files"][n]["sha256"] == h for n, h in EXPECTED_SFT.items())
print("OK: these are the team's fine-tuning files." if same else
      "The files differ from the team's build: check that step 9 printed OK.")
print("fine-tuning files ->", os.path.abspath(SFT_DIR))

## 12. Baseline: the base model without fine-tuning

This step runs **Qwen2.5-7B-Instruct as is** (no fine-tuning) on the 170 test rows. It uses the
same system prompt and guidance passages the fine-tuned model will see, then scores the answers
with `eval/run.py`. These are the "before" numbers that the fine-tuned model is compared with.

- **Needs a GPU.** Choose **Runtime → Change runtime type → T4 GPU** before **Run all**. On a CPU
  runtime this step prints a message and skips, and the rest of the notebook still runs.
- The model loads in 4-bit (about 5.5 GB of GPU memory). Answers are greedy, so a rerun gives
  the same output.
- Answers are saved as they are generated, so if Colab disconnects, run the cell again to resume.
  A rerun in the same session reuses the loaded model.
- Rows are sorted by length, and a batch that does not fit in GPU memory is retried one row at
  a time, so long complaints do not stop the run.
- Outputs: `baseline/predictions_base.jsonl` (one `{"id", "output"}` per test row) and
  `baseline/report_base.json` (all metrics with 95% confidence intervals).
- Set `LIMIT = 16` for a quick check first. The scorer then counts the missing rows as wrong.

In [ ]:
# ---- baseline settings ----
RUN_BASELINE = True                       # False: skip this section
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # the model we fine-tune, here used as is
BATCH_SIZE = 8                            # lower it if the GPU runs out of memory
MAX_NEW_TOKENS = 400                      # a full answer is about 150 to 250 tokens
LIMIT = None                              # a number runs only the first N test rows (quick check)

import importlib.util, json, os, subprocess, sys, time

BASE_DIR = os.path.join(OUT_DIR, "baseline")
PRED_PATH = os.path.join(BASE_DIR, "predictions_base.jsonl")
REPORT_PATH = os.path.join(BASE_DIR, "report_base.json")
TEST_PATH = os.path.join(REPO_DIR, "data", "sft", "cfpb_test.jsonl")


def generate_batch(model, tok, batch):
    import torch
    prompts = [tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True)
               for r in batch]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)


def generate_all(model, tok, rows):
    """Greedy answers for rows not yet in PRED_PATH; appends as it goes, so a rerun resumes."""
    import torch
    done = set()
    if os.path.exists(PRED_PATH):
        with open(PRED_PATH, encoding="utf-8") as f:
            done = {json.loads(line)["id"] for line in f if line.strip()}
    # Similar lengths in a batch means less padding; the longest prompts come last.
    todo = sorted((r for r in rows if r["id"] not in done),
                  key=lambda r: len(r["messages"][1]["content"]))
    print(f"{len(done)} already done, {len(todo)} to go")
    start = time.time()
    for i in range(0, len(todo), BATCH_SIZE):
        batch = todo[i:i + BATCH_SIZE]
        try:
            texts = generate_batch(model, tok, batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()          # batch too long for the GPU: one row at a time
            texts = []
            for r in batch:
                texts += generate_batch(model, tok, [r])
                torch.cuda.empty_cache()
        with open(PRED_PATH, "a", encoding="utf-8") as f:
            for r, t in zip(batch, texts):
                f.write(json.dumps({"id": r["id"], "output": t.strip()}, ensure_ascii=False) + "\n")
        n = i + len(batch)
        print(f"  {n}/{len(todo)}  {(time.time() - start) / 60:.1f} min")


import torch

if not RUN_BASELINE:
    print("skipped: RUN_BASELINE is False")
elif not torch.cuda.is_available():
    print("skipped: this section needs a GPU. Choose Runtime -> Change runtime type -> T4 GPU, "
          "then run all again.")
else:
    if importlib.util.find_spec("bitsandbytes") is None:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "bitsandbytes"], check=True)
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    os.makedirs(BASE_DIR, exist_ok=True)
    with open(TEST_PATH, encoding="utf-8") as f:
        test_rows = [json.loads(line) for line in f]
    if LIMIT:
        test_rows = test_rows[:LIMIT]
    print(f"GPU: {torch.cuda.get_device_name(0)} | model: {BASE_MODEL} (4-bit) | "
          f"test rows: {len(test_rows)}")

    if "base_model" not in globals():         # a rerun reuses the model already in memory
        base_tok = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")
        if base_tok.pad_token is None:
            base_tok.pad_token = base_tok.eos_token
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, device_map="auto",
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_compute_dtype=torch.float16))
        base_model.eval()
    generate_all(base_model, base_tok, test_rows)

    cmd = [sys.executable, os.path.join(REPO_DIR, "eval", "run.py"), PRED_PATH,
           "--test", TEST_PATH, "--out", REPORT_PATH]
    run = subprocess.run(cmd, capture_output=True, text=True)
    print(run.stdout, run.stderr)
    run.check_returncode()
    print("predictions ->", os.path.abspath(PRED_PATH))
    print("report      ->", os.path.abspath(REPORT_PATH))

## 13. Download the files (Colab only)

In [ ]:
if IN_COLAB and not SAVE_TO_DRIVE:
    import shutil
    from google.colab import files as colab_files
    shutil.make_archive("cfpb_dataset", "zip", OUT_DIR)
    colab_files.download("cfpb_dataset.zip")
else:
    print("files are in", os.path.abspath(OUT_DIR))